In [2]:
import pandas as pd

In [ ]:
# Run the Genome Nexus Pipeline .sh file seperate, to get the OncoKB Annotation on the mutational artifcat removed maf file (signature analyzer output).
# Apply at OncoKB to receive a token which you can use in the indicated field below. Without token, the OncoKB annotation will be skipped!!!

  #!/bin/bash
docker run --rm \
  -e GENOMENEXUS_BASE=https://grch38.genomenexus.org \
  -v ${PWD}:/wd \
  genomenexus/gn-annotation-pipeline:latest \
  java \
  -Doncokb.token= <ENTER YOUR TOKEN HERE> \
  -Dgenomenexus.enrichment_fields=annotation_summary,my_variant_info,polyphen,sift,mutation_assessor,oncokb,nucleotide_context \
  -jar annotationPipeline.jar \
  --filename /wd/snv_artefact_filtered_modified.maf \
  --output-filename /wd/GenomeNexus_output_enriched.txt \
  --isoform-override mskcc


In [ ]:
# Load output from OncoKB annotated 
df_OnkoKB = pd.read_csv('mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/2026_02_27_GenomeNexus_output_enriched.txt', sep='\t', comment='#')

/var/folders/54/_6vs7vnn4433pfg0vbsbtjgw0000gn/T/ipykernel_68436/65047969.py:1: DtypeWarning: Columns (0: oncokb_highestSensitiveLevel) have mixed types. Specify dtype option on import or set low_memory=False.
  df_OnkoKB = pd.read_csv('mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/2026_02_27_GenomeNexus_output_enriched.txt', sep='\t', comment='#')


In [ ]:
# Load SNV maf file to control if OncoKB output has same size 
df_snv_artefact_filter = pd.read_csv('mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/snv_artefact_filtered_modified.maf', sep='\t')
df_snv_artefact_filter

,Hugo_Symbol,Chromosome,Start_Position,End_Position,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,t_alt_count,t_ref_count,n_alt_count,n_ref_count,Tumor_Sample_Barcode,Protein_Change
0,MTOR,1,11114411,11114412,Frame_Shift_Ins,INS,-,-,TG,8,59,0,24,09T02-09N01-MT-union,NaN
1,AADACL3,1,12727616,12727616,3'UTR,SNP,G,G,T,20,48,0,25,09T02-09N01-MT-union,NaN
2,IGSF21,1,18376374,18376374,Missense_Mutation,SNP,G,G,T,15,52,0,15,09T02-09N01-MT-union,NaN
3,CATSPER4,1,26202660,26202660,3'UTR,SNP,G,G,C,14,49,0,28,09T02-09N01-MT-union,NaN
4,ZDHHC18,1,26855096,26855096,3'UTR,SNP,C,C,T,18,52,0,28,09T02-09N01-MT-union,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24853,LY6K,8,142700625,142700625,Missense_Mutation,SNP,G,G,A,8,77,0,52,WGS_15_2-WGS_16_2,p.R33Q
24854,TMEM215,9,32787577,32787577,3'UTR,DEL,C,C,-,6,71,0,45,WGS_15_2-WGS_16_2,NaN
24855,TUBB4B,9,137242974,137242974,Silent,SNP,G,G,A,5,63,0,51,WGS_15_2-WGS_16_2,p.K252K
24856,CNKSR2,X,21595036,21595036,Silent,SNP,C,C,T,4,45,0,29,WGS_15_2-WGS_16_2,p.C631C


In [8]:
# Control if GenomeNexus output contains all mutations which were there before
df_OnkoKB.shape[0] == df_snv_artefact_filter.shape[0]

True

In [ ]:
# Shortly check values of oncokb_oncogenic column
df_OnkoKB["oncokb_oncogenic"].value_counts()

oncokb_oncogenic
Unknown             855
Likely Oncogenic    105
Oncogenic             9
Inconclusive          2
Likely Neutral        1
Name: count, dtype: int64

In [ ]:
# Genes with most oncogenic and likley oncogenic mutations
df_OnkoKB[(df_OnkoKB['oncokb_oncogenic']=='Oncogenic') | (df_OnkoKB['oncokb_oncogenic']=='Likely Oncogenic')]['Hugo_Symbol'].value_counts()

Hugo_Symbol
TP53        42
RB1         16
ATRX        12
PTEN         3
NF2          3
KDM6A        2
TSC2         2
KMT2D        2
PIK3R1       2
PDGFRB       1
EP300        1
ETV6         1
PIK3CA       1
SUFU         1
PTPN1        1
CDKN2A       1
KMT2B        1
PPP2R2A      1
TP63         1
ATXN2        1
ARID1A       1
PML          1
ERCC3        1
MED12        1
KMT2C        1
CREBBP       1
TET1         1
LATS2        1
STAG2        1
SETD2        1
ESCO2        1
PTPRT        1
TGFBR2       1
SUZ12        1
FLCN         1
LRP1B        1
ARHGAP35     1
IDH1         1
H3-3A        1
Name: count, dtype: int64

In [ ]:
# Filter to only OncoKB "oncogenic" or "likely oncogenic"
df_OnkoKB_only = df_OnkoKB[(df_OnkoKB['oncokb_oncogenic']=='Oncogenic') | (df_OnkoKB['oncokb_oncogenic']=='Likely Oncogenic')] #| (df_OnkoKB['oncokb_oncogenic']=='Unknown') # was first included but now we remove it so that we are consistent

In [ ]:
# Safe file for heatmap plot
df_OnkoKB_only.to_csv('mutsig_tool_files_MutSig2CV_standalone_macOS/SNVs_filtered_signatureanalyser/2026_03_01_OncoKB_OncoKB_oncogenic_or_likely_oncogenic_Mutations_only.maf', sep='\t', index=False)
